# Marine Buoy Weather Forecasting
### Data Science for Business — NOAA Buoys (1980–2023)

This notebook describes step by step the analysis developed for forecasting marine weather variables and identifying buoy similarity patterns.

**Targets:** `WVHT` (significant wave height, m) and `WTMP` (water temperature, °C)  
**Forecasting buoy:** 42002 — Gulf of Mexico (26°N, 93°W)  
**Clustering buoys:** 42002, 42001, 42039 (Gulf of Mexico) + 46042 (Pacific, Monterey)  
**Dataset:** [`Qdrant/NOAA-Buoy`](https://huggingface.co/datasets/Qdrant/NOAA-Buoy) + NDBC

**Index:**
1. Setup
2. Data loading (`load.py`)
3. Data cleaning (`clean.py`)
4. Exploratory Data Analysis
5. Feature engineering & temporal split (`features.py`)
6. Regression models & comparison (`models.py`)
7. AutoML with FLAML (`automl.py`)
8. Buoy clustering (`clustering.py`)
9. Web Application
10. Conclusions


## 1. Setup


In [ ]:
import sys, json, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches

ROOT = Path('.').resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

PROCESSED = ROOT / 'data' / 'processed'
MODELS    = ROOT / 'models'
CLUSTER   = PROCESSED / 'clustering'
RESULTS   = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)

BLUE, ORANGE, GREEN = '#2563eb', '#ea580c', '#16a34a'
GRAY   = '#93c5fd'
COLORS = [BLUE, ORANGE, GREEN, '#9333ea', '#ca8a04', '#1098ad']
print('ROOT:', ROOT)

**Setup:** imports, path configuration, color palette for plots. Results are saved to `results/`.


---
## 2. Data Loading (`load.py`)

**Primary dataset:** [`Qdrant/NOAA-Buoy`](https://huggingface.co/datasets/Qdrant/NOAA-Buoy) — buoy 42002, hourly data 1980–2023.

**Problem:** `load_dataset('Qdrant/NOAA-Buoy')` fails with `DatasetGenerationCastError` due to inconsistent CSV columns across monthly files. **Solution:** download the processed `.parquet` files directly via `huggingface_hub`.

**Extra stations** downloaded from NDBC for clustering analysis:
- 42001, 42039 — Gulf of Mexico
- 46042 — Pacific Ocean (Monterey, CA)

**Output:** `data/raw/buoys_all.csv` with all 4 stations.


In [ ]:
raw_f = ROOT / 'data' / 'raw' / 'buoys_all.csv'
if raw_f.exists():
    raw = pd.read_csv(raw_f, parse_dates=['timestamp'])
    print('Raw dataset:')
    print(f'  Total rows: {len(raw):,}')
    print(f'  Stations: {sorted(raw["station_id"].astype(str).unique())}')
    print(f'  Columns: {list(raw.columns)}')
    for st, g in raw.groupby('station_id'):
        print(f'  {st}: {len(g):,} rows [{g["timestamp"].min().date()} → {g["timestamp"].max().date()}]')
else:
    print('Raw file not found — run load.py first')

**This block:** loads the raw CSV and shows rows per station and time coverage. Buoy 42002 has the longest history (1980–2023); the NDBC stations cover 2015–2023.


---
## 3. Data Cleaning (`clean.py`)

Two operations applied to all stations:

**Physical bounds filtering:** values outside plausible ranges are replaced with NaN (sensor errors or sentinel values). Examples: WVHT > 30m, WTMP < -5°C or > 40°C, PRES outside 800–1100 hPa.

**Hourly resampling** with `resample('1h').mean()`: creates a **regular time grid** with exactly one observation per hour. This is essential for lag features — without a regular grid, lag1 would not mean '1 hour ago'.

**Output:** `data/processed/buoys_clean.csv` (all stations) + `buoy_42002_clean.csv` (primary station only).


In [ ]:
df_all = pd.read_csv(PROCESSED / 'buoys_clean.csv', parse_dates=['timestamp'])
df_all['station_id'] = df_all['station_id'].astype(str)
print('All stations after cleaning:')
for st, g in df_all.groupby('station_id'):
    miss_wvht = g['WVHT'].isna().mean()
    miss_wtmp = g['WTMP'].isna().mean()
    print(f'  {st}: {len(g):,} rows | WVHT missing={miss_wvht:.1%} | WTMP missing={miss_wtmp:.1%}')

df = pd.read_csv(PROCESSED / 'buoy_42002_clean.csv', parse_dates=['timestamp'])
print(f'\nBuoy 42002: {len(df):,} rows, {df["timestamp"].dt.year.nunique()} years')
display(df[['WVHT','WTMP','WSPD','PRES','ATMP']].describe().round(3))

**This block:** shows missing rates per station after cleaning and descriptive statistics for buoy 42002.


---
## 4. Exploratory Data Analysis

Before building models, we analyze the structure of the time series to understand trends, seasonality, and distributions. Key observations:
- **WVHT** shows strong annual seasonality: higher waves in winter, calmer in summer
- **WTMP** shows opposite seasonality: warmer water in summer, cooler in winter
- Both variables have high short-term autocorrelation — the current value is a strong predictor of the next


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6))
df.set_index('timestamp')['WVHT'].dropna().plot(ax=axes[0], lw=0.3, color=BLUE, alpha=0.8)
axes[0].set_title('WVHT — Significant Wave Height (m) — Buoy 42002 (1980–2023)', fontsize=12)
axes[0].set_ylabel('m'); axes[0].grid(alpha=0.3)
df.set_index('timestamp')['WTMP'].dropna().plot(ax=axes[1], lw=0.3, color=ORANGE, alpha=0.8)
axes[1].set_title('WTMP — Water Temperature (°C) — Buoy 42002 (1980–2023)', fontsize=12)
axes[1].set_ylabel('°C'); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS / '01_time_series.png', dpi=120, bbox_inches='tight')
plt.show()

**This block:** full time series 1980–2023. Annual seasonality is clearly visible in both variables. Saved as `results/01_time_series.png`.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
df['WVHT'].dropna().plot(kind='hist', bins=60, ax=axes[0][0], color=BLUE)
axes[0][0].set_title('WVHT distribution'); axes[0][0].set_xlabel('m')
df['WTMP'].dropna().plot(kind='hist', bins=60, ax=axes[0][1], color=ORANGE)
axes[0][1].set_title('WTMP distribution'); axes[0][1].set_xlabel('°C')
df.assign(month=df['timestamp'].dt.month).groupby('month')['WVHT'].mean().plot(
    kind='bar', ax=axes[1][0], color=BLUE)
axes[1][0].set_title('WVHT mean by month (higher in winter)')
axes[1][0].set_xlabel('Month'); axes[1][0].set_ylabel('m')
df.assign(month=df['timestamp'].dt.month).groupby('month')['WTMP'].mean().plot(
    kind='bar', ax=axes[1][1], color=ORANGE)
axes[1][1].set_title('WTMP mean by month (warmer in summer)')
axes[1][1].set_xlabel('Month'); axes[1][1].set_ylabel('°C')
plt.suptitle('EDA — Distribution and Seasonality', fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS / '02_eda.png', dpi=120, bbox_inches='tight')
plt.show()

**This block:** distributions and monthly averages confirming seasonality. Motivates the use of cyclic calendar features (sin/cos). Saved as `results/02_eda.png`.


---
## 5. Feature Engineering & Temporal Split (`features.py`)

The time series problem is transformed into a supervised learning problem:
- **y** = target value at time `t+1` (what we want to predict)
- **X** = everything known up to time `t` (no future information)

**Features built (no data leakage):**
- **Lag features:** `target_lag1`, `lag2`, `lag3`, `lag6`, `lag12`, `lag24` — exploit autocorrelation
- **Rolling stats:** `target_roll3`, `roll6`, `roll24` — short-term trend
- **Exogenous variables:** current + lag1 of WSPD, PRES, ATMP, etc.
- **Cyclic calendar:** `hour_sin/cos` (daily cycle), `doy_sin/cos` (annual seasonality)

**Temporal split 70% / 15% / 15% — NO shuffle:**
```
[========= TRAIN 70% =========][=== VAL 15% ===][=== TEST 15% ===]
        1980 → ~2013               ~2013→2018        2018→2023
```
Random shuffle would cause data leakage (future data in training). The model is selected on **validation** — test is used only once at the end.


In [ ]:
import joblib
for target in ['WVHT', 'WTMP']:
    train = pd.read_csv(PROCESSED / f'train_42002_{target}.csv', parse_dates=['timestamp'])
    val   = pd.read_csv(PROCESSED / f'val_42002_{target}.csv',   parse_dates=['timestamp'])
    test  = pd.read_csv(PROCESSED / f'test_42002_{target}.csv',  parse_dates=['timestamp'])
    feat  = [c for c in train.columns if c not in ('timestamp','y')]
    print(f'\n=== {target} — {len(feat)} features ===')
    print(f'  Train: {len(train):>7,} rows [{train["timestamp"].min().date()} → {train["timestamp"].max().date()}]')
    print(f'  Val:   {len(val):>7,} rows [{val["timestamp"].min().date()} → {val["timestamp"].max().date()}]')
    print(f'  Test:  {len(test):>7,} rows [{test["timestamp"].min().date()} → {test["timestamp"].max().date()}]')
    print(f'  Sample features: {feat[:6]}')

**This block:** shows the exact date ranges and sizes of each split. Confirms no overlap between train, val and test.


---
## 6. Regression Models & Comparison (`models.py`)

Four strategies trained and compared for each target (WVHT and WTMP):

| Model | Type | Key property |
|---|---|---|
| **Persistence** | Baseline | ŷ(t+1) = y(t). No training. Minimum reference. |
| **Ridge** | Linear + L2 | Prevents overfitting with correlated features (lags). Needs StandardScaler. |
| **Random Forest** | Bagging | Builds many trees on random subsamples, averages predictions. Reduces variance. |
| **Gradient Boosting** | Boosting | Builds trees sequentially on residuals. Reduces bias. Generally most accurate. |

**Metrics:** MAE (mean absolute error), RMSE (penalizes large errors more), R² (proportion of variance explained).  
**Model selection:** best on **validation RMSE** — test set is never used for decisions.


In [ ]:
metrics_f = MODELS / 'metrics.json'
if not metrics_f.exists():
    print('Run models.py first')
else:
    m = json.loads(metrics_f.read_text())
    for target, info in m.items():
        print(f'\n=== {target} === Best on val RMSE: {info["best"].upper()}')
        rows = [{'model':n,
            'val MAE':v['val']['MAE'],'val RMSE':v['val']['RMSE'],'val R²':v['val']['R2'],
            'test MAE':v['test']['MAE'],'test RMSE':v['test']['RMSE'],'test R²':v['test']['R2']}
            for n,v in info['metrics'].items()]
        display(pd.DataFrame(rows).sort_values('val RMSE').reset_index(drop=True))
        p = info['metrics']['persistence']['test']['RMSE']
        b = info['metrics'][info['best']]['test']['RMSE']
        print(f'  → Improvement over persistence: {(p-b)/p*100:.1f}%')

**This block:** loads metrics from `models/metrics.json` and shows the comparison table sorted by validation RMSE. The improvement percentage shows how much the best model beats the persistence baseline.


In [ ]:
if metrics_f.exists():
    m = json.loads(metrics_f.read_text())
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for row, target in enumerate(['WVHT','WTMP']):
        info = m[target]
        names = list(info['metrics'].keys())
        for col, (split, met) in enumerate([('val','RMSE'),('val','R2'),('test','RMSE')]):
            vals   = [info['metrics'][n][split][met] for n in names]
            colors = [BLUE if n==info['best'] else GRAY for n in names]
            axes[row][col].bar(names, vals, color=colors)
            axes[row][col].set_title(f'{target} — {split} {met}', fontsize=10)
            axes[row][col].tick_params(axis='x', rotation=25, labelsize=8)
            if met=='R2': axes[row][col].set_ylim(0,1)
            axes[row][col].grid(axis='y', alpha=0.3)
    plt.suptitle('Model Comparison — blue = best on validation RMSE', fontsize=12)
    plt.tight_layout()
    plt.savefig(RESULTS / '03_model_comparison.png', dpi=120, bbox_inches='tight')
    plt.show()

**This block:** bar charts comparing all models across validation and test metrics. Blue bar = selected model. Saved as `results/03_model_comparison.png`.


In [ ]:
if metrics_f.exists():
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    for i, target in enumerate(['WVHT','WTMP']):
        test_f   = PROCESSED / f'test_42002_{target}.csv'
        bundle_f = MODELS / f'best_{target}.joblib'
        if not test_f.exists() or not bundle_f.exists(): continue
        test   = pd.read_csv(test_f, parse_dates=['timestamp'])
        bundle = joblib.load(bundle_f)
        X = test[bundle['features']].values.astype(float)
        if bundle['scaler']: X = bundle['scaler'].transform(X)
        yhat  = bundle['model'].predict(X)
        ytrue = test['y'].values
        pers  = test[f'{target}_t0'].values
        sl = slice(-500, None)
        axes[i].plot(test['timestamp'].iloc[sl], ytrue[sl], label='actual', lw=1.5, color='#1e293b')
        axes[i].plot(test['timestamp'].iloc[sl], yhat[sl], label=f'predicted ({bundle["name"]})', lw=1, color=BLUE, alpha=0.85)
        axes[i].plot(test['timestamp'].iloc[sl], pers[sl], label='persistence', lw=0.8, color=ORANGE, linestyle='--', alpha=0.7)
        axes[i].set_title(f'{target} — actual vs predicted vs persistence (last 500 test points)', fontsize=11)
        axes[i].legend(fontsize=9); axes[i].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS / '04_predictions_vs_actual.png', dpi=120, bbox_inches='tight')
    plt.show()

**This block:** actual vs predicted vs persistence on the last 500 test points. The model (blue) tracks the actual (black) more closely than persistence (orange dashed). Saved as `results/04_predictions_vs_actual.png`.


---
## 7. AutoML with FLAML (`automl.py`)

**FLAML** (Fast and Lightweight AutoML, Microsoft) automatically searches for the best model and hyperparameters within a time budget.

**Configuration:**
```python
automl.fit(
    task='regression', metric='rmse',
    time_budget=60,       # 60 seconds
    eval_method='cv', split_type='time',  # temporal CV — no leakage
    n_splits=5,
)
```
`split_type='time'` ensures each CV fold respects chronological order — consistent with the manual split.


In [ ]:
automl_f = MODELS / 'metrics_automl.json'
if not automl_f.exists():
    print('Run automl.py first')
else:
    a = json.loads(automl_f.read_text())
    print('AutoML Results (FLAML, 60s budget):')
    for target, info in a.items():
        print(f'\n  {target}: best={info["best_estimator"]}')
        print(f'    Val  → RMSE={info["metrics"]["val"]["RMSE"]} R²={info["metrics"]["val"]["R2"]}')
        print(f'    Test → RMSE={info["metrics"]["test"]["RMSE"]} R²={info["metrics"]["test"]["R2"]}')

**This block:** shows the algorithm automatically selected by FLAML and its metrics. FLAML explores LightGBM, XGBoost, Random Forest, Ridge and returns the best configuration found within 60 seconds.


In [ ]:
if metrics_f.exists() and automl_f.exists():
    m = json.loads(metrics_f.read_text())
    a = json.loads(automl_f.read_text())
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for i, target in enumerate(['WVHT','WTMP']):
        info  = m[target]
        names = list(info['metrics'].keys()) + [f'AutoML\n({a[target]["best_estimator"]})']
        rmses = ([info['metrics'][n]['test']['RMSE'] for n in info['metrics']]
                 + [a[target]['metrics']['test']['RMSE']])
        colors = [ORANGE if 'AutoML' in n else (BLUE if n.split('\n')[0].strip()==info['best'] else GRAY) for n in names]
        axes[i].bar(names, rmses, color=colors)
        axes[i].set_title(f'{target} — Test RMSE\nblue=best sklearn, orange=AutoML')
        axes[i].tick_params(axis='x', rotation=20, labelsize=8)
        axes[i].grid(axis='y', alpha=0.3)
    plt.suptitle('Full comparison: classical models vs AutoML', fontsize=12)
    plt.tight_layout()
    plt.savefig(RESULTS / '05_automl_comparison.png', dpi=120, bbox_inches='tight')
    plt.show()

**This block:** compares classical models and AutoML in a single chart. Orange bar = AutoML, blue = best sklearn model. Saved as `results/05_automl_comparison.png`.


---
## 8. Buoy Clustering (`clustering.py`)

KMeans clustering is applied to identify **which buoys have similar meteorological behavior** and how those similarities change over time.

Each buoy/period is described by: **mean + std of WVHT, WTMP, WSPD, PRES, ATMP** (10 features).

**Pipeline:**
1. **StandardScaler** — normalize features (KMeans uses Euclidean distance, sensitive to scale)
2. **KMeans** — try k from 2 to 8
3. **Silhouette Score** — automatically select the best k (higher = better separated clusters)
4. **Elbow Plot** — inertia vs k as alternative method
5. **PCA 2D** — reduce to 2 components for visualization

### Static Clustering (buoy profiles)
Each point = **one buoy**, described by its overall profile across the entire available period.
Shows which buoys have similar long-term meteorological conditions.

### Dynamic Clustering (yearly profiles)
Each point = **one buoy in one year**.
Shows how each buoy's regime shifts year by year — identifies anomalous years and long-term changes.


In [ ]:
static_f = CLUSTER / 'static.json'
if not static_f.exists():
    print('Run clustering.py first')
else:
    s   = json.loads(static_f.read_text())
    tab = pd.read_csv(CLUSTER / 'static.csv')
    print('Static Clustering (buoy profiles):')
    print(f'  k={s["k"]}  silhouette={s["silhouette"]}  stations={s["n_stations"]}')
    for c, stations in s['clusters'].items():
        print(f'  Cluster {c}: {stations}')

**This block:** shows static clustering results — which buoys are grouped together based on their overall meteorological profile. Typically the Pacific buoy (46042) is separated from the Gulf of Mexico buoys.


In [ ]:
if static_f.exists():
    s   = json.loads(static_f.read_text())
    tab = pd.read_csv(CLUSTER / 'static.csv')
    fig = plt.figure(figsize=(15, 5))
    gs  = gridspec.GridSpec(1, 3, figure=fig)

    ax0 = fig.add_subplot(gs[0])
    for c in sorted(tab['cluster'].unique()):
        sub = tab[tab['cluster']==c]
        ax0.scatter(sub['pca_x'], sub['pca_y'], label=f'Cluster {c}',
                    s=120, color=COLORS[c % len(COLORS)], zorder=3)
        for _, r in sub.iterrows():
            ax0.annotate(str(r['station_id']), (r['pca_x'], r['pca_y']),
                         fontsize=9, xytext=(5,5), textcoords='offset points')
    ax0.set_xlabel('PCA 1'); ax0.set_ylabel('PCA 2')
    ax0.set_title(f'Buoy profiles — PCA 2D\nk={s["k"]} sil={s["silhouette"]}')
    ax0.legend(); ax0.grid(alpha=0.3)

    ax1 = fig.add_subplot(gs[1])
    sil_k = sorted([int(k) for k in s['silhouette_by_k']])
    sil_v = [s['silhouette_by_k'][str(k)] for k in sil_k]
    ax1.bar([f'k={k}' for k in sil_k], sil_v,
            color=[BLUE if k==s['k'] else GRAY for k in sil_k])
    ax1.set_title('Silhouette Score by k\nhigher = better, blue = selected')
    ax1.set_ylim(0, max(sil_v)*1.2 if sil_v else 1); ax1.grid(axis='y', alpha=0.3)

    ax2 = fig.add_subplot(gs[2])
    inertia_k = sorted([int(k) for k in s['inertia_by_k']])
    inertia_v = [s['inertia_by_k'][str(k)] for k in inertia_k]
    ax2.plot([f'k={k}' for k in inertia_k], inertia_v, 'o-', color=BLUE, lw=2, markersize=8)
    ax2.set_title('Elbow Plot — Inertia by k'); ax2.grid(alpha=0.3)

    plt.suptitle('Static Clustering — Buoy Profiles (each point = 1 buoy)', fontsize=12)
    plt.tight_layout()
    plt.savefig(RESULTS / '06_static_clustering.png', dpi=120, bbox_inches='tight')
    plt.show()

**This block:** PCA scatter (each point = one buoy, labeled with station ID), silhouette plot, and elbow plot for the static clustering. Saved as `results/06_static_clustering.png`.


In [ ]:
dyn_f = CLUSTER / 'dynamic.json'
if not dyn_f.exists():
    print('Run clustering.py first')
else:
    d    = json.loads(dyn_f.read_text())
    dtab = pd.read_csv(CLUSTER / 'dynamic.csv')
    print('Dynamic Clustering (yearly profiles):')
    print(f'  k={d["k"]}  silhouette={d["silhouette"]}  points={d["n_points"]}')
    print('  Sequences:')
    for st, seq in d['sequences'].items():
        print(f'    {st}: {" → ".join(map(str, seq["clusters"]))} ({seq["changes"]} changes)')

**This block:** shows dynamic clustering sequences — how each buoy's regime changed year by year. A change in cluster indicates the buoy moved to a different meteorological regime in that year.


In [ ]:
if dyn_f.exists():
    d    = json.loads(dyn_f.read_text())
    dtab = pd.read_csv(CLUSTER / 'dynamic.csv')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for c in sorted(dtab['cluster'].unique()):
        sub = dtab[dtab['cluster']==c]
        axes[0].scatter(sub['pca_x'], sub['pca_y'], label=f'Cluster {c}',
                        color=COLORS[c % len(COLORS)], s=60, zorder=3)
        for _, r in sub.iterrows():
            axes[0].annotate(f"{r['station_id']}\n{int(r['year'])}",
                             (r['pca_x'], r['pca_y']),
                             fontsize=6, xytext=(3,3), textcoords='offset points')
    axes[0].set_xlabel('PCA 1'); axes[0].set_ylabel('PCA 2')
    axes[0].set_title(f'Dynamic Clustering — PCA 2D\neach point = 1 buoy in 1 year\nk={d["k"]} sil={d["silhouette"]}')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    stations_list = sorted(d['sequences'].keys())
    for idx, st in enumerate(stations_list):
        seq = d['sequences'][st]
        for yr, cl in zip(seq['years'], seq['clusters']):
            axes[1].bar(yr, 1, bottom=idx, color=COLORS[cl % len(COLORS)], width=0.8, alpha=0.85)
    axes[1].set_yticks(range(len(stations_list)))
    axes[1].set_yticklabels(stations_list)
    axes[1].set_title('Regime per year per buoy\n(color = cluster)')
    axes[1].set_xlabel('Year')
    handles = [mpatches.Patch(color=COLORS[c], label=f'Cluster {c}')
               for c in range(d['k'])]
    axes[1].legend(handles=handles, loc='upper right', fontsize=8)
    axes[1].grid(axis='x', alpha=0.2)

    plt.suptitle('Dynamic Clustering — How regimes vary year by year across all buoys', fontsize=12)
    plt.tight_layout()
    plt.savefig(RESULTS / '07_dynamic_clustering.png', dpi=120, bbox_inches='tight')
    plt.show()

**This block:** dynamic clustering visualization. Left: PCA scatter with (buoy, year) labels. Right: stacked bar chart showing the regime of each buoy for each year — a change in color means a regime transition. Saved as `results/07_dynamic_clustering.png`.


---
## 9. Web Application

The trained models are exposed via a web application with two FastAPI components running in a single Docker container managed by **supervisord**:

### Backend — REST API (port 8080)
Loads `.joblib` bundles, runs inference, reads clustering outputs. Uses **Pydantic** for data validation.

| Endpoint | Description |
|---|---|
| `GET /api/health` | Service status |
| `GET /api/stations` | List of available buoy stations |
| `GET /api/predict` | WVHT + WTMP forecast at t+1h |
| `GET /api/comparison` | MAE/RMSE/R² for all models |
| `GET /api/clusters/static` | Static buoy clustering |
| `GET /api/clusters/dynamic` | Dynamic yearly clustering |

### Frontend — Web Interface (port 8000)
Calls the backend via `httpx`, renders pages with **Jinja2** templates and **Chart.js** for interactive charts. Generated HTML templates and CSS stylesheet.

| Page | Content |
|---|---|
| `/` | Dashboard: current WVHT + WTMP predictions |
| `/prediction` | Dropdown to select station + time series chart + forecast point |
| `/comparison` | Metrics table: all models + AutoML |
| `/clustering` | PCA scatter + silhouette + elbow + dynamic regime chart |

### Docker
```bash
# Pull and run (models included)
docker pull ghcr.io/brusca01/marine-buoy-forecasting:latest
docker run --rm -p 8000:8000 -p 8080:8080 ghcr.io/brusca01/marine-buoy-forecasting:latest
```
Frontend → http://localhost:8000 | Backend → http://localhost:8080/docs


---
## 10. Conclusions

### What was done
1. **Downloaded** buoy 42002 from `Qdrant/NOAA-Buoy` (HF) + stations 42001, 42039, 46042 from NDBC
2. **Cleaned** data: physical bounds + hourly resampling for a regular time grid
3. **Analyzed** seasonality, distributions and autocorrelation (EDA)
4. **Built** lag features, rolling stats, cyclic calendar features — no data leakage
5. **Trained** persistence, Ridge, Random Forest, Gradient Boosting on 70/15/15 temporal split
6. **Compared** with AutoML (FLAML, 60s, temporal CV)
7. **Clustered** buoys by meteorological profile (static) and tracked regime changes year by year (dynamic)
8. **Deployed** as FastAPI backend + frontend in Docker

### Key findings
- **WVHT** is highly predictable due to strong short-term autocorrelation (R² > 0.98 on test)
- **WTMP** has extremely high autocorrelation at 1-hour horizon (R² > 0.999)
- The **persistence baseline** is strong on both — ML models add value by capturing nonlinear patterns
- **AutoML** automatically finds XGBoost/LightGBM, often competitive with manually tuned models
- **Static clustering** separates the Pacific buoy (46042) from the Gulf of Mexico buoys — different wave and temperature regimes
- **Dynamic clustering** reveals how each buoy's annual regime shifts over decades, identifying anomalous years

**Dataset:** https://huggingface.co/datasets/Qdrant/NOAA-Buoy

### Run the application
```bash
docker pull ghcr.io/brusca01/marine-buoy-forecasting:latest
docker run --rm -p 8000:8000 -p 8080:8080 ghcr.io/brusca01/marine-buoy-forecasting:latest
```
Frontend → http://localhost:8000 | Backend → http://localhost:8080/docs

